# Case Citation — forward citation counts in fixed windows

How often a case is cited, within 3, 5, 10 years of its decision and over all time. Primary
key: `case_id`. Reads only `cache/case_graph.npz`, written by **case_metadata** — run that first.

## The metric
For case *F* decided in year *y*, `C_w` counts the citing cases *A* with

$$0 \le \text{year}(A) - y \le w$$

so a same-year citation counts and a citation that predates the decision does not. `C_all`
drops the upper bound only. The window is on the **citing** case's decision year, which is the
only date either endpoint has.

Unlike the patent analogue there is no examiner / applicant split and no application-stage
citation: a case citation is a case citation, so where `patent_citation.parquet` carries 45
columns this carries five.

## Output
`Case law/output/case_citation.parquet` — `case_id, C_3, C_5, C_10, C_all`

One row per case, including cases never cited (`C_* = 0`) — 1,391,837 of them. Filtering them
out here would make every downstream mean conditional on being cited, which is a different
quantity.

## What is dropped
An edge is counted only if **both** endpoints carry a decision year. All 5,179,698 cases do
(the leading-four-digit parse in `cl_common` recovers 100%, against 93.0% for a strict
`YYYY-MM-DD` parse), so in this snapshot nothing is lost to missing dates — the count below
says so explicitly rather than leaving it to be assumed.

In [1]:
%%time
import os, sys, gc, time
import numpy as np, pandas as pd
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/Case law')
import cl_common as cl
OUT_FP = cl.out('case_citation.parquet')
WINDOWS = [3, 5, 10, -1]                      # -1 = all time
SFX = {3: '_3', 5: '_5', 10: '_10', -1: '_all'}
cl.preflight('case_citation')

c_from, c_to, year, uni = cl.load_graph()     # c_from CITES c_to
n = len(uni)
yF, yT = year[c_from], year[c_to]
lag = yF.astype(np.int32) - yT.astype(np.int32)
dated = (yF > 0) & (yT > 0)
ok = dated & (lag >= 0)
print(f'edges {len(c_from):,}')
print(f'  both endpoints dated   {int(dated.sum()):,}  ({dated.mean()*100:.2f}%)')
print(f'  negative lag dropped   {int((dated & (lag < 0)).sum()):,}')
print(f'  counted                {int(ok.sum()):,}')

case law : /project/jevans/Dawoon/Science of Science/Case law
output   : /project/jevans/Dawoon/Science of Science/Case law/output
cache    : /project/jevans/Dawoon/Science of Science/Case law/cache

  case_citation               OK
graph cache present: /project/jevans/Dawoon/Science of Science/Case law/cache/case_graph.npz
edges 47,519,638
  both endpoints dated   47,519,638  (100.00%)
  negative lag dropped   0
  counted                47,519,638


## Count

`np.bincount` over the cited endpoint, once per window. Fully vectorised — no per-case loop —
so the whole table is four passes over the 47.5M-edge lag array.

In [2]:
%%time
cols = {'case_id': uni}
for w in WINDOWS:
    m = ok if w == -1 else (ok & (lag <= w))
    cols[f'C{SFX[w]}'] = np.bincount(c_to[m], minlength=n).astype(np.int32)
cit = pd.DataFrame(cols)
cit.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(cit):,} rows, {len(cit.columns)} cols, '
      f'{os.path.getsize(OUT_FP)/1e6:.0f} MB)\n')
for w in WINDOWS:
    c = cit[f'C{SFX[w]}']
    print(f'  C{SFX[w]:<5} mean {c.mean():7.3f}   median {int(c.median()):>4}   '
          f'p99 {int(c.quantile(.99)):>5}   max {c.max():>7,}   zero {int((c==0).sum()):>9,}')
display(cit.sort_values('C_all', ascending=False).head(5))

WROTE /project/jevans/Dawoon/Science of Science/Case law/output/case_citation.parquet  (5,179,698 rows, 5 cols, 41 MB)

  C_3    mean   2.142   median    1   p99    22   max  12,723   zero 2,557,911
  C_5    mean   3.053   median    1   p99    30   max  14,822   zero 2,223,784
  C_10   mean   4.715   median    1   p99    46   max  22,770   zero 1,856,373
  C_all  mean   9.174   median    3   p99    83   max  69,085   zero 1,391,837


,case_id,C_3,C_5,C_10,C_all
3419338,6206897,2855,5609,15586,69085
3419376,6207800,2963,5830,15783,62498
3419273,6204802,2515,4143,8582,42081
3418300,6182418,1581,3018,7390,36237
3418314,6182629,402,780,1469,35495
